<div style="text-align: center; background-color: #5A96E3; font-family: 'Trebuchet MS', Arial, sans-serif; color: white; padding: 20px; font-size: 40px; font-weight: bold; border-radius: 0 0 0 0; box-shadow: 0px 6px 8px rgba(0, 0, 0, 0.2);">
  Stage 04 - RAG implementation 📌
</div>

## I. Import libraries

In [1]:
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.embeddings import GPT4AllEmbeddings
from langchain.schema import HumanMessage
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

## II. Config environment

In [ ]:
os.environ["DEEPSEEK_API_KEY"] = ""

## III. RAG for question answering

In [3]:
llm = ChatDeepSeek(model="deepseek-chat")
embedding_model = GPT4AllEmbeddings()

In [4]:
client = QdrantClient(path="../qdrant_initial_db")

collection_name = "candidates"

vectorstore = Qdrant(
    client=client,
    collection_name=collection_name,
    embeddings=embedding_model,
    content_payload_key="text"
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5864\3436551038.py:5: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use :class:`~QdrantVectorStore` instead.
  vectorstore = Qdrant(


In [14]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

template = """
Bạn là một hệ thống HR AI. Hãy trả lời dựa trên thông tin ứng viên trong cơ sở dữ liệu.

Câu hỏi: {question}

Thông tin tìm được từ cơ sở dữ liệu:
{context}

Trả lời ngắn gọn và rõ ràng:
"""
QA_PROMPT = PromptTemplate(input_variables=["question", "context"], template=template)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": QA_PROMPT},
)

query = "Ứng viên JR Sabado có những kĩ năng gì"
# result = qa_chain.run(query)

# print("🔍 Kết quả:", result)

docs = retriever.get_relevant_documents(query)

print("=== 🔎 Retrieved Documents ===")
for i, d in enumerate(docs, 1):
    print(f"\n--- Candidate {i} ---")

    print(f"Raw text: {d.page_content}")

=== 🔎 Retrieved Documents ===

--- Candidate 1 ---
Raw text: 

--- Candidate 2 ---
Raw text: Name: JR Sabado, Email: sabadotweetie@gmail.com, Skills: Javascript, Typescript, PHP, Java, VB.Net, SQL, NodeJS, AngularJS, Angular, jQuery, Jasmine, Mocha, Bootstrap, Foundation, Angular Material, Codeigniter, HTML, CSS, SCSS, SVN, Git, Azure Devops, Visual Studio Code, Wordpress, Heroku, Experience: Software Engineer at Infor, PSSC, Inc. | Application Development Analyst at Accenture, Inc. | R&D PHP Developer at Gameloft Philippines | Software Development Engineer at Allied Telesis Labs, Inc. | Web Developer Intern at Granton World Philippines

--- Candidate 3 ---
Raw text: Name: Nithin Narayanan, Email: nithin19.n@gmail.com, Skills: Strong Experience in Object Oriented Programming and Software Implementation best practice, Knowledge in website development, Excellent analytical thinking and solution provider mind set, Excellent in product development and life cycle management, Excellent in Te